# Regressão Logística

In [155]:
import numpy as np

Problema: classificação de e-mails como spam

In [156]:
# links | palavra "grátis" | classificação (y)
# y = 1 -> spam
# y = 0 -> não spam
dados = np.array([[0, 0, 0],
                 [3, 1, 1],
                 [1, 0, 0],
                 [5, 1, 1]])

print(dados)

[[0 0 0]
 [3 1 1]
 [1 0 0]
 [5 1 1]]


$$ z = b_1 x_1 + b_2 x_2 + (...) + b_j x_j + b_0 = b^T x $$

- $ x_j $: valor da j-ésima característica
- $ b_j $: peso aprendido para a j-ésima característica ($w_j$ e $b_j$ são iguais)
- $ b_0 $: viés (intercepto)
- $ z \in \R $: pontuação que ainda não é probabilidade

Função logística (sigmoide)

$$ \sigma(z) = \frac{1}{1 + e^{-z}}$$

- Recebe $ z \in \R $
- Retorna $\sigma(z) \in (0, 1) $

In [157]:
def sigmoide(z):
    return 1 / (1 + np.e**(-z))

Exemplo:

$\sigma(2) \approx 0.881 $

Probabilidades: 88.1% para a classe 1, 11.9% para a classe 0.

Ajuste dos parâmetros -> Entropia cruzada binária

Não usamos o MSE porque os erros ficam extremamente pequenos (os erros podem ser altos, mas o MSE fica pequeno)

$$ L(p, y) = \begin{cases} -ln(p), & \text{se } y = 1, \\ -ln(1 - p), & \text{se } y = 0. \end{cases} $$

- para $y = 1$ -> queremos $p$ próximo de 1
- para $y = 0$ -> queremos $p$ próximo de 0


$$ L(p, y) = -[y ln(p) + (1-y) ln(1 - p)] $$

$$ J(w,b) = \frac{1}{m} \sum_{i=1}^{m} L(p^{(i)}, y^{(i)}) $$

Atualização dos pesos por descida do gradiente

$$ w_j \leftarrow w_j - \alpha \frac{\partial  J}{\partial  w_j} $$

$$ b \leftarrow b - \alpha \frac{\partial  J}{\partial  b} $$

$$ \frac{\partial  J}{\partial  w_j} = \frac{1}{m} \sum_{i=1}^{m} (p^{(i)} - y^{(i)}) x_j^{(i)}$$

$$ \frac{\partial  J}{\partial  b} = \frac{1}{m} \sum_{i=1}^{m} (p^{(i)} - y^{(i)})$$

No final das contas:

$$ \frac{1}{m} (p - y) x_j $$

para $b_0$, não há o termo '$x_j$'

In [158]:
def regressao_logistica(x, y, learning_rate = 0.1):

    m, n = x.shape

    pesos = np.zeros(n)
    
    b0 = 0

    epocas = 1000

    for epoca in range(epocas):
        z = np.dot(x, pesos) + b0
        
        p = sigmoide(z)

        # calculando gradientes de cada coeficiente e do intercepto
        dw = (1 / m) * np.dot(x.T, (p - y))

        db = (1 / m) * np.sum (p - y)

        pesos -= learning_rate * dw

        b0 -= learning_rate * db

    return b0, pesos

In [159]:
x = dados[:, :-1]
y = dados[:, -1]

b0, coefs = regressao_logistica(x, y)

print(f"Intecepto: {b0}")
print(f"Coeficientes: ")

for i in range (len(coefs)):
    print(f"x{i}: {coefs[i]}")

Intecepto: -4.419592072184113
Coeficientes: 
x0: 1.9422025056814036
x1: 2.0220224631832613


Limiar de decisão

$$ \hat{y} = \begin{cases} 1, & \text{se } \hat{p} \ge t , \\ 0, & \text{se } \hat{p} < t. \end{cases} $$

- Limiar menor: tende a detectar mais positivos, com mais falsos alarmes
- Limiar maior: tende a reduzir falsos alarmes, mas pode perder positivos

In [160]:
def limiar_de_decisao(p, t):
    if p >= t:
        return 1
    return 0

In [161]:
def classificacao(x, pesos, limiar):

    coefs = pesos[1:]

    z = pesos[0] + np.dot(x, coefs)

    sig = sigmoide(z)

    classe = limiar_de_decisao(sig, limiar)

    return classe

In [162]:
pesos = np.array([b0, coefs[0], coefs[1]])

print("Classe 0 -> não spam")
print("Classe 1 -> spam")

teste1 = classificacao(np.array([0, 1]), pesos, 0.5)
print(f"Teste 1: classe {teste1}")

teste2 = classificacao(np.array([4, 0]), pesos, 0.5)
print(f"Teste 2: classe {teste2}")

Classe 0 -> não spam
Classe 1 -> spam
Teste 1: classe 0
Teste 2: classe 1


### Métricas de Avaliação ###

In [163]:
def avaliar_modelo(X, y, pesos, limiar):

    fp = 0
    fn = 0
    vp = 0
    vn = 0
    acertos = 0

    for i in range (X.shape[0]):
        
        y_prev = classificacao(X[i, :], pesos, limiar)

        if y[i] == y_prev:
            if y_prev == 1:
                vp += 1
            else:
                vn += 1

            acertos += 1
        else:
            if y_prev == 1:
                fp += 1
            else:
                fn += 1

    return fp, fn, vp, vn, acertos

In [164]:
# dados gerados por IA

X = np.array([
    [0, 0],
    [3, 1],
    [1, 0],
    [5, 1],
    [0, 0],
    [1, 0],
    [2, 0],
    [0, 1],
    [1, 1],
    [2, 0],
    [3, 0],
    [1, 0],
    [0, 0],
    [2, 1],
    [4, 1],
    [6, 1],
    [7, 1],
    [8, 1],
    [5, 0],
    [6, 0],
    [4, 1],
    [9, 1],
    [3, 1],
    [7, 0]
])

y = np.array([
    0, 1, 0, 1,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1
])

quantidade_de_testes = len(y)

In [165]:
fp, fn, vp, vn, acertos = avaliar_modelo(X, y, pesos, 0.5)

In [166]:
print(f"Falso positivo: {fp}")
print(f"Falso negativo: {fn}")
print(f"Verdadeiro positivo: {vp}")
print(f"Verdadeiro negativo: {vn}")

Falso positivo: 2
Falso negativo: 0
Verdadeiro positivo: 12
Verdadeiro negativo: 10


$$ \text{Acurácia simples} = \frac{\text{acertos}}{\text{quantidade de testes}} $$

In [167]:
def acuracia_simples(acertos, quantidade):
    return acertos / quantidade

print(acuracia_simples(acertos, quantidade_de_testes))

0.9166666666666666


Matriz de confusão

In [168]:
def print_matriz_confusao(fp, fn, vp, vn):
    total = vp + vn + fp + fn

    print(f"Exemplo com {total} casos")
    print("-" * 52)
    print(f"{'':16} | {'Predição':^31}")
    print(f"{'Real':^16} | {'Positivo (1)':^14} | {'Negativo (0)':^14}")
    print("-" * 52)
    print(f"{'Positivo (1)':<16} | {f'VP = {vp}':^14} | {f'FN = {fn}':^14}")
    print(f"{'Negativo (0)':<16} | {f'FP = {fp}':^14} | {f'VN = {vn}':^14}")
    print("-" * 52)
    print("\n▶ VP: verdadeiro positivo.      ▶ FP: falso positivo.")
    print("▶ VN: verdadeiro negativo.      ▶ FN: falso negativo.")

print_matriz_confusao(fp, fn, vp, vn)

Exemplo com 24 casos
----------------------------------------------------
                 |            Predição            
      Real       |  Positivo (1)  |  Negativo (0) 
----------------------------------------------------
Positivo (1)     |    VP = 12     |     FN = 0    
Negativo (0)     |     FP = 2     |    VN = 10    
----------------------------------------------------

▶ VP: verdadeiro positivo.      ▶ FP: falso positivo.
▶ VN: verdadeiro negativo.      ▶ FN: falso negativo.


$$ \text{Acurácia(Accuracy)} = \frac{VP + VN}{VP + VN + FP + FN} $$

In [169]:
def acuracia(vp, vn, fp, fn):
    return (vp + vn) / (vp + vn + fp + fn)

print(acuracia(vp, vn, fp, fn))

0.9166666666666666


$$ \text{Precisão(Precision)} = \frac{VP}{VP + FP} $$

In [170]:
def precisao (vp, fp):
    return vp / (vp + fp)

precision = precisao(vp, fp)
print(precision)

0.8571428571428571


$$ \text{Revocação(Recall)} = \frac{VP}{VP + FN} $$

In [171]:
def recall(vp, fn):
    return vp / (vp + fn)

rec = recall(vp, fn)
print(rec)

1.0


$$ \text{F1-score} = 2 . \frac{\text{Precisão . Recall}}{\text{Precisão + Recall}}$$

In [172]:
def f1_score(precision, recall):
    return 2 * ((precision * recall) / (precision + recall))

print(f1_score(precision, rec))

0.923076923076923
